# Staging QA Notebook

Notebook de controle pour verifier rapidement si les tables `stg_*` sont saines, avec un focus particulier sur le decodage de `tickLower` / `tickUpper` dans les evenements LP.

Objectifs:
- verifier `raw -> staging` sur `Mint` et `Burn`
- confirmer que les ticks ne sont pas corrompus
- verifier l'ordre `tick_lower < tick_upper`
- verifier l'alignement au `tick_spacing`
- sortir rapidement des samples suspects si un probleme reapparait


## Prerequis

Si besoin, decommenter et executer:

```python
%pip install pandas clickhouse-connect plotly jupyterlab
```


In [ ]:
import os
from textwrap import dedent

import clickhouse_connect
from IPython.display import display

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'pandas is required. Run: %pip install pandas clickhouse-connect plotly jupyterlab'
    ) from exc

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 200)


In [ ]:
# Defaults local-first pour clickhouse_local; surcharge possible via variables d'environnement.
CLICKHOUSE_HOST = os.getenv('CLICKHOUSE_HOST', 'localhost')
CLICKHOUSE_PORT = int(os.getenv('CLICKHOUSE_PORT', '8124'))
CLICKHOUSE_USER = os.getenv('CLICKHOUSE_USER', 'default')
CLICKHOUSE_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '1234')
CLICKHOUSE_DATABASE = os.getenv('CLICKHOUSE_DATABASE', 'analytics')

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

print(f'Connected to ClickHouse database={CLICKHOUSE_DATABASE} on {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}')


In [ ]:
TIME_COLUMNS = ['date', 'block_time', 'deployment_ts']

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    for column in TIME_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column])
    return df

def query_df(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    df = client.query_df(sql, parameters=parameters or {})
    df = normalize_df(df)
    print(f'{len(df):,} rows')
    return df


## 1. Inventaire rapide du staging


In [ ]:
staging_inventory = query_df(dedent(
    '''
    SELECT
        table_name,
        count() AS rows,
        min(block_time) AS min_block_time,
        max(block_time) AS max_block_time,
        uniqExact(pool) AS distinct_pools
    FROM (
        SELECT 'stg_swaps' AS table_name, pool, block_time FROM stg_swaps
        UNION ALL
        SELECT 'stg_mints' AS table_name, pool, block_time FROM stg_mints
        UNION ALL
        SELECT 'stg_burns' AS table_name, pool, block_time FROM stg_burns
    )
    GROUP BY table_name
    ORDER BY table_name
    '''
))
staging_inventory


In [ ]:
available_pools = query_df(dedent(
    '''
    SELECT
        pool,
        min(toDate(block_time)) AS min_date,
        max(toDate(block_time)) AS max_date,
        count() AS swap_rows
    FROM stg_swaps
    GROUP BY pool
    ORDER BY swap_rows DESC, pool
    '''
))
available_pools


In [ ]:
POOL = available_pools.iloc[0]['pool'] if not available_pools.empty else None
EXPECTED_TICK_SPACING = 10
INT24_MIN = -887272
INT24_MAX = 887272

print('POOL:', POOL)
print('EXPECTED_TICK_SPACING:', EXPECTED_TICK_SPACING)


## 2. Raw -> staging: est-ce que des lignes LP sont rejetees ?

Si le garde-fou `toInt32OrNull(tickLower/tickUpper)` filtre encore des lignes corrompues, on doit le voir ici.


In [ ]:
raw_vs_staging = query_df(dedent(
    '''
    SELECT
        'mint' AS table_name,
        (SELECT count() FROM uniswap_usdc_weth_mint) AS raw_rows,
        (SELECT count() FROM stg_mints) AS staged_rows,
        raw_rows - staged_rows AS dropped_rows
    UNION ALL
    SELECT
        'burn' AS table_name,
        (SELECT count() FROM uniswap_usdc_weth_burn) AS raw_rows,
        (SELECT count() FROM stg_burns) AS staged_rows,
        raw_rows - staged_rows AS dropped_rows
    '''
))
raw_vs_staging


In [ ]:
raw_decode_audit = query_df(dedent(
    '''
    SELECT
        'mint' AS table_name,
        count() AS raw_rows,
        countIf(toInt32OrNull(tickLower) IS NULL) AS null_tick_lower,
        countIf(toInt32OrNull(tickUpper) IS NULL) AS null_tick_upper,
        countIf(amount0 IS NULL OR amount1 IS NULL) AS null_amounts
    FROM uniswap_usdc_weth_mint
    UNION ALL
    SELECT
        'burn' AS table_name,
        count() AS raw_rows,
        countIf(toInt32OrNull(tickLower) IS NULL) AS null_tick_lower,
        countIf(toInt32OrNull(tickUpper) IS NULL) AS null_tick_upper,
        countIf(amount0 IS NULL OR amount1 IS NULL) AS null_amounts
    FROM uniswap_usdc_weth_burn
    '''
))
raw_decode_audit


## 3. Sanity check ticks dans le staging


In [ ]:
tick_summary = query_df(dedent(
    '''
    SELECT
        table_name,
        count() AS rows,
        min(tick_lower) AS min_tick_lower,
        max(tick_lower) AS max_tick_lower,
        min(tick_upper) AS min_tick_upper,
        max(tick_upper) AS max_tick_upper,
        countIf(tick_lower >= tick_upper) AS bad_order,
        countIf(tick_lower < %(int24_min)s OR tick_lower > %(int24_max)s) AS lower_outside_int24_range,
        countIf(tick_upper < %(int24_min)s OR tick_upper > %(int24_max)s) AS upper_outside_int24_range,
        countIf(modulo(tick_lower, %(tick_spacing)s) != 0) AS lower_not_spacing,
        countIf(modulo(tick_upper, %(tick_spacing)s) != 0) AS upper_not_spacing
    FROM (
        SELECT 'stg_mints' AS table_name, tick_lower, tick_upper FROM stg_mints
        UNION ALL
        SELECT 'stg_burns' AS table_name, tick_lower, tick_upper FROM stg_burns
    )
    GROUP BY table_name
    ORDER BY table_name
    '''
), {'tick_spacing': EXPECTED_TICK_SPACING, 'int24_min': INT24_MIN, 'int24_max': INT24_MAX})
tick_summary


In [ ]:
width_summary = query_df(dedent(
    '''
    SELECT
        table_name,
        min(width_ticks) AS min_width_ticks,
        quantileExact(0.25)(width_ticks) AS p25_width_ticks,
        quantileExact(0.5)(width_ticks) AS median_width_ticks,
        quantileExact(0.75)(width_ticks) AS p75_width_ticks,
        max(width_ticks) AS max_width_ticks,
        uniqExact(width_ticks) AS distinct_widths
    FROM (
        SELECT 'stg_mints' AS table_name, tick_upper - tick_lower AS width_ticks FROM stg_mints
        UNION ALL
        SELECT 'stg_burns' AS table_name, tick_upper - tick_lower AS width_ticks FROM stg_burns
    )
    GROUP BY table_name
    ORDER BY table_name
    '''
))
width_summary


In [ ]:
top_widths = query_df(dedent(
    '''
    SELECT
        table_name,
        width_ticks,
        count() AS rows
    FROM (
        SELECT 'stg_mints' AS table_name, tick_upper - tick_lower AS width_ticks FROM stg_mints
        UNION ALL
        SELECT 'stg_burns' AS table_name, tick_upper - tick_lower AS width_ticks FROM stg_burns
    )
    GROUP BY table_name, width_ticks
    ORDER BY table_name, rows DESC, width_ticks
    LIMIT 20 BY table_name
    '''
))
top_widths


## 4. Samples suspects si un probleme reapparait

Ces cellules doivent idealement renvoyer 0 ligne. Si ce n'est pas le cas, on a un vrai sujet de decode ou de nettoyage.


In [ ]:
bad_raw_tick_samples = query_df(dedent(
    '''
    SELECT
        'mint' AS table_name,
        block_number,
        log_index,
        tx_hash,
        tickLower AS raw_tick_lower,
        tickUpper AS raw_tick_upper
    FROM uniswap_usdc_weth_mint
    WHERE toInt32OrNull(tickLower) IS NULL OR toInt32OrNull(tickUpper) IS NULL
    LIMIT 25
    UNION ALL
    SELECT
        'burn' AS table_name,
        block_number,
        log_index,
        tx_hash,
        tickLower AS raw_tick_lower,
        tickUpper AS raw_tick_upper
    FROM uniswap_usdc_weth_burn
    WHERE toInt32OrNull(tickLower) IS NULL OR toInt32OrNull(tickUpper) IS NULL
    LIMIT 25
    '''
))
bad_raw_tick_samples


In [ ]:
bad_staging_tick_samples = query_df(dedent(
    '''
    SELECT
        'stg_mints' AS table_name,
        block_number,
        log_index,
        tx_hash,
        tick_lower,
        tick_upper,
        tick_upper - tick_lower AS width_ticks
    FROM stg_mints
    WHERE tick_lower >= tick_upper
       OR modulo(tick_lower, %(tick_spacing)s) != 0
       OR modulo(tick_upper, %(tick_spacing)s) != 0
       OR tick_lower < %(int24_min)s OR tick_lower > %(int24_max)s
       OR tick_upper < %(int24_min)s OR tick_upper > %(int24_max)s
    LIMIT 25
    UNION ALL
    SELECT
        'stg_burns' AS table_name,
        block_number,
        log_index,
        tx_hash,
        tick_lower,
        tick_upper,
        tick_upper - tick_lower AS width_ticks
    FROM stg_burns
    WHERE tick_lower >= tick_upper
       OR modulo(tick_lower, %(tick_spacing)s) != 0
       OR modulo(tick_upper, %(tick_spacing)s) != 0
       OR tick_lower < %(int24_min)s OR tick_lower > %(int24_max)s
       OR tick_upper < %(int24_min)s OR tick_upper > %(int24_max)s
    LIMIT 25
    '''
), {'tick_spacing': EXPECTED_TICK_SPACING, 'int24_min': INT24_MIN, 'int24_max': INT24_MAX})
bad_staging_tick_samples


## 5. Focus optionnel sur un pool

Si plusieurs pools sont charges plus tard dans le meme environnement local, ces cellules permettent de re-checker seulement la pool active.


In [ ]:
pool_tick_summary = query_df(dedent(
    '''
    SELECT
        table_name,
        count() AS rows,
        min(tick_lower) AS min_tick_lower,
        max(tick_lower) AS max_tick_lower,
        min(tick_upper) AS min_tick_upper,
        max(tick_upper) AS max_tick_upper
    FROM (
        SELECT 'stg_mints' AS table_name, pool, tick_lower, tick_upper FROM stg_mints
        UNION ALL
        SELECT 'stg_burns' AS table_name, pool, tick_lower, tick_upper FROM stg_burns
    )
    WHERE pool = %(pool)s
    GROUP BY table_name
    ORDER BY table_name
    '''
), {'pool': POOL}) if POOL else pd.DataFrame()
pool_tick_summary


## 6. Swap QA

Ici on verifie les colonnes critiques de `stg_swaps`: `tick`, `sqrtPriceX96` raw, `amount0`, `amount1`, `liquidity`, et la price derivee.


In [ ]:
raw_swap_vs_staging = query_df(dedent(
    '''
    SELECT
        (SELECT count() FROM uniswap_usdc_weth_swap) AS raw_rows,
        (SELECT count() FROM stg_swaps) AS staged_rows,
        raw_rows - staged_rows AS dropped_rows
    '''
))
raw_swap_vs_staging


In [ ]:
raw_swap_decode_audit = query_df(dedent(
    '''
    SELECT
        count() AS raw_rows,
        countIf(amount0 IS NULL OR amount1 IS NULL) AS null_amounts,
        countIf(sqrtPriceX96 IS NULL) AS null_sqrt_price,
        countIf(toInt32OrNull(tick) IS NULL) AS null_tick,
        countIf(toUInt256OrNull(sqrtPriceX96) IS NULL) AS invalid_sqrt_parse,
        countIf(toUInt256OrNull(sqrtPriceX96) = 0) AS zero_sqrt_price
    FROM uniswap_usdc_weth_swap
    '''
))
raw_swap_decode_audit


In [ ]:
swap_summary = query_df(dedent(
    '''
    SELECT
        count() AS rows,
        min(tick) AS min_tick,
        max(tick) AS max_tick,
        min(price_token1_in_token0) AS min_price,
        max(price_token1_in_token0) AS max_price,
        min(liquidity) AS min_liquidity,
        max(liquidity) AS max_liquidity,
        min(abs(toFloat64(amount0))) AS min_abs_amount0,
        quantileTDigest(0.5)(abs(toFloat64(amount0))) AS median_abs_amount0,
        quantileTDigest(0.99)(abs(toFloat64(amount0))) AS p99_abs_amount0,
        min(abs(toFloat64(amount1))) AS min_abs_amount1,
        quantileTDigest(0.5)(abs(toFloat64(amount1))) AS median_abs_amount1,
        quantileTDigest(0.99)(abs(toFloat64(amount1))) AS p99_abs_amount1
    FROM stg_swaps
    '''
))
swap_summary


In [ ]:
swap_invariants = query_df(dedent(
    '''
    WITH
        (SELECT countIf(amount0 = '0' OR amount1 = '0') FROM uniswap_usdc_weth_swap) AS raw_zero_leg_swaps
    SELECT
        countIf(tick < %(int24_min)s OR tick > %(int24_max)s) AS tick_outside_int24,
        countIf(price_token1_in_token0 <= 0) AS non_positive_price,
        countIf(liquidity < 0) AS negative_liquidity,
        countIf((amount0 > 0 AND amount1 > 0) OR (amount0 < 0 AND amount1 < 0)) AS same_sign_amounts,
        countIf(amount0 = 0 OR amount1 = 0) AS staged_zero_leg_swaps,
        raw_zero_leg_swaps,
        countIf(amount0 = 0 OR amount1 = 0) - raw_zero_leg_swaps AS precision_loss_zero_leg_swaps
    FROM stg_swaps
    '''
), {'int24_min': INT24_MIN, 'int24_max': INT24_MAX})
swap_invariants


In [ ]:
swap_price_tick_consistency = query_df(dedent(
    '''
    SELECT
        count() AS rows,
        quantileTDigest(0.5)(abs(price_token1_in_token0 / (pow(1.0001, -tick) * 1000000000000.0) - 1)) AS p50_rel_err_vs_tick,
        quantileTDigest(0.99)(abs(price_token1_in_token0 / (pow(1.0001, -tick) * 1000000000000.0) - 1)) AS p99_rel_err_vs_tick,
        max(abs(price_token1_in_token0 / (pow(1.0001, -tick) * 1000000000000.0) - 1)) AS max_rel_err_vs_tick
    FROM stg_swaps
    '''
))
swap_price_tick_consistency


In [ ]:
swap_precision_loss_samples = query_df(dedent(
    '''
    SELECT
        s.tx_hash,
        s.block_number,
        s.log_index,
        r.amount0 AS raw_amount0,
        r.amount1 AS raw_amount1,
        s.amount0 AS stg_amount0,
        s.amount1 AS stg_amount1,
        s.price_token1_in_token0,
        s.tick
    FROM stg_swaps s
    INNER JOIN uniswap_usdc_weth_swap r
        ON s.tx_hash = r.tx_hash
       AND s.block_number = r.block_number
       AND s.log_index = toUInt64(r.log_index)
    WHERE (s.amount0 = 0 OR s.amount1 = 0)
      AND NOT (r.amount0 = '0' OR r.amount1 = '0')
    ORDER BY s.block_number, s.log_index
    LIMIT 25
    '''
))
swap_precision_loss_samples


### Lecture rapide pour les swaps

- `dropped_rows = 0` et `null_* = 0` => le raw swap est complet et parse correctement.
- `tick_outside_int24 = 0`, `non_positive_price = 0`, `negative_liquidity = 0`, `same_sign_amounts = 0` => pas de valeur protocolaire aberrante evidente.
- `precision_loss_zero_leg_swaps > 0` => le staging arrondit certaines petites jambes a zero, ce qui signale plutot un sujet de precision qu'un raw corrompu.
- `rel_err_vs_tick` autour de `0.01%` est normal: le `tick` est discret alors que la price derivee de `sqrtPriceX96` est plus precise.


## Interpretation rapide

Le staging est probablement sain si:
- `dropped_rows = 0` sur `Mint` et `Burn`
- `null_tick_lower = 0` et `null_tick_upper = 0` dans le raw
- `bad_order = 0`
- `lower_not_spacing = 0` et `upper_not_spacing = 0`
- aucune ligne dans les cellules de samples suspects

Si un de ces checks casse, on sait tout de suite si le probleme vient du raw, du decoding, ou du nettoyage staging.
